<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [9]</a>'.</span>

In [1]:
# ONE-CELL SUPERCHARGED NOTEBOOK with LIGHT STACKING: high-ROI EDA + style-first features + elastic-net LR
# + word/char/numeric specialist models -> meta-logistic stack + bucketed thresholds
# Target: push F1 toward ~0.9 with only numpy/pandas/scikit-learn/matplotlib.

In [2]:
# =========================
# 0) Imports & globals
# =========================
import os, re, io, gzip, math, string, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.base import BaseEstimator, TransformerMixin, clone
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.preprocessing import StandardScaler
from sklearn.feature_extraction.text import TfidfVectorizer, HashingVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV, calibration_curve
from sklearn.decomposition import TruncatedSVD
from sklearn.metrics import f1_score, classification_report, precision_recall_curve, roc_auc_score

warnings.filterwarnings("ignore")
RANDOM_STATE = 42
JOBS = -1
BASE_DIR = ""  # set to your Drive path if needed
TRAIN_PATH = BASE_DIR + "train.csv"
VAL_PATH   = BASE_DIR + "val.csv"
TEST_PATH  = BASE_DIR + "test.csv"

In [3]:
# =========================
# 1) EDA helpers
# =========================
def _print_section(title): 
    print("\n" + "="*92 + f"\n{title}\n" + "="*92)

def _safe_hist(ax, data, bins=50, title="", xlabel="", ylabel="Count", logy=False):
    data = np.asarray(pd.Series(data).values, dtype=float)
    data = data[~np.isnan(data)]
    if data.size == 0:
        ax.text(0.5, 0.5, "No data", ha="center", va="center"); return
    ax.hist(data, bins=bins, edgecolor="black", linewidth=0.5)
    ax.set_title(title); ax.set_xlabel(xlabel); ax.set_ylabel(ylabel)
    ax.grid(True, linestyle=":", alpha=0.6)
    if logy: ax.set_yscale("log")

def _psi(a, b, bins=20):
    a = np.asarray(a, float); b = np.asarray(b, float)
    a = a[~np.isnan(a)]; b = b[~np.isnan(b)]
    if a.size < 10 or b.size < 10: return np.nan
    edges = np.quantile(a, np.linspace(0,1,bins+1))
    edges[0] -= 1e-6; edges[-1] += 1e-6
    a_hist,_ = np.histogram(a, bins=edges); b_hist,_ = np.histogram(b, bins=edges)
    a_p = a_hist / max(1,a_hist.sum()); b_p = b_hist / max(1,b_hist.sum())
    a_p = np.clip(a_p, 1e-6, 1.0); b_p = np.clip(b_p, 1e-6, 1.0)
    return np.sum((a_p - b_p) * np.log(a_p / b_p))

In [4]:
# =========================
# 2) Load data
# =========================
train = pd.read_csv(TRAIN_PATH)
val   = pd.read_csv(VAL_PATH)
test  = pd.read_csv(TEST_PATH)
assert {'id','text','label'}.issubset(train.columns)
assert {'id','text','label'}.issubset(val.columns)
assert {'id','text'}.issubset(test.columns)
train['label'] = train['label'].astype(int)
val['label']   = val['label'].astype(int)

In [5]:
# =========================
# 3) Compact EDA snapshot
# =========================
_print_section("Basic shape & schema")
print("Train:", train.shape, " Val:", val.shape, " Test:", test.shape)
_print_section("Missingness & empties")
for name, df in [("train", train), ("val", val), ("test", test)]:
    miss = df.isnull().mean().round(4)
    empty_text = (df['text'].fillna("").str.strip().eq("")).mean().round(4)
    print(f"{name}: missing\n{miss}\n{name}: empty text -> {empty_text}\n")
_print_section("Duplicates & overlaps")
for name, df in [("train", train), ("val", val), ("test", test)]:
    print(f"{name}: dup id={df['id'].duplicated().sum()}, dup text={df['text'].duplicated().sum()}")
tset = set(train['text'].astype(str)); vset = set(val['text'].astype(str)); teset = set(test['text'].astype(str))
print("train∩val:", len(tset & vset), " train∩test:", len(tset & teset), " val∩test:", len(vset & teset))
_print_section("Label priors")
print("Train label prop:\n", (train['label'].value_counts()/len(train)).round(3))
print("Val   label prop:\n", (val['label'].value_counts()/len(val)).round(3))
_print_section("Shift canary: PSI on char length (train vs val)")
psi = _psi(train['text'].astype(str).str.len(), val['text'].astype(str).str.len())
print(f"PSI={psi:.3f}  (heuristic: <0.1 small, 0.1–0.25 moderate, >0.25 large)")


Basic shape & schema
Train: (319071, 3)  Val: (56792, 3)  Test: (60743, 2)

Missingness & empties


train: missing
text     0.0
label    0.0
id       0.0
dtype: float64
train: empty text -> 0.0

val: missing
text     0.0
label    0.0
id       0.0
dtype: float64
val: empty text -> 0.0

test: missing
id      0.0
text    0.0
dtype: float64
test: empty text -> 0.0


Duplicates & overlaps


train: dup id=0, dup text=0
val: dup id=0, dup text=0
test: dup id=0, dup text=762


train∩val: 0  train∩test: 0  val∩test: 52

Label priors
Train label prop:
 label
0    0.708
1    0.292
Name: count, dtype: float64
Val   label prop:
 label
1    0.507
0    0.493
Name: count, dtype: float64

Shift canary: PSI on char length (train vs val)


PSI=0.016  (heuristic: <0.1 small, 0.1–0.25 moderate, >0.25 large)


In [6]:
# =========================
# 4) Style-first normalization
# =========================
_digit = re.compile(r"\d+")
def normalize_text(s: str) -> str:
    if not isinstance(s, str): return ""
    s = s.replace("’","'").replace("“","\"").replace("”","\"").replace("—","-").replace("–","-")
    s = _digit.sub("<num>", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s

In [7]:
# =========================
# 5) Stylometric feature block
# =========================
FUNCTION_WORDS = set("""
a an the and but or if because as until while of at by for with about against
between into through during before after above below to from up down in out on
off over under again further then once here there when where why how all any
both each few more most other some such no nor not only own same so than too
very can will just should now
""".split())

def gzip_ratio(s: str) -> float:
    if not s: return 0.0
    raw = s.encode("utf-8","ignore")
    out = io.BytesIO()
    with gzip.GzipFile(fileobj=out, mode="w") as f:
        f.write(raw)
    return len(out.getvalue()) / max(1, len(raw))

def word_shapes(words):
    def shape(w):
        w2 = re.sub(r"\d", "d", w)
        w2 = ''.join('X' if c.isupper() else 'x' if c.islower() else 'd' if c=='d' else c for c in w2)
        w2 = re.sub(r"[^Xxd]+", "_", w2)
        return w2
    return [shape(w) for w in words]

class StylisticFeatures(BaseEstimator, TransformerMixin):
    def __init__(self):
        self.stopwords = FUNCTION_WORDS
        self.punct = set(string.punctuation)
    @staticmethod
    def _safe(a,b): return float(a)/float(b) if b else 0.0
    def fit(self, X, y=None): return self
    def transform(self, X):
        feats = []
        for t in X:
            s = t if isinstance(t, str) else ""
            s = s.strip()
            chars = len(s)
            words = re.findall(r"\b\w+\b", s.lower())
            n_words = len(words); uniq = set(words); n_uniq = len(uniq)
            avg_word_len = np.mean([len(w) for w in words]) if n_words else 0.0
            long_word_ratio = self._safe(sum(len(w)>=7 for w in words), n_words)
            ttr  = self._safe(n_uniq, n_words)
            # hapax
            hapax = 0.0
            if n_words:
                freq = {}
                for w in words: freq[w] = freq.get(w,0)+1
                hapax = self._safe(sum(c==1 for c in freq.values()), n_words)
            punct_cnt = sum(1 for ch in s if ch in self.punct)
            digit_cnt = sum(1 for ch in s if ch.isdigit())
            upper_cnt = sum(1 for ch in s if ch.isupper())
            space_cnt = sum(1 for ch in s if ch.isspace())
            func_cnt = sum(1 for w in words if w in self.stopwords)
            # entropy
            if chars:
                counts = {}
                for ch in s: counts[ch] = counts.get(ch,0)+1
                ent = 0.0
                for c in counts.values():
                    p = c/chars; ent -= p*math.log(p+1e-12, 2)
            else:
                ent = 0.0
            # repetition/uniqueness
            def rep_ratio(seq, n):
                if len(seq) < n: return 0.0, 0.0
                grams = [tuple(seq[i:i+n]) for i in range(len(seq)-n+1)]
                total = len(grams); uniq = len(set(grams)); repeats = total - uniq
                return self._safe(repeats,total), self._safe(uniq,total)
            rep1, uniq1 = rep_ratio(words, 1)
            rep2, uniq2 = rep_ratio(words, 2)
            rep3, uniq3 = rep_ratio(words, 3)
            # run-lengths
            max_tok_run, cur = 1, 1
            for i in range(1, n_words):
                if words[i] == words[i-1]: cur += 1
                else:
                    if cur > max_tok_run: max_tok_run = cur
                    cur = 1
            if n_words: max_tok_run = max(max_tok_run, cur)
            max_punct_run, cur = 0, 0
            for ch in s:
                if ch in self.punct: cur += 1
                else:
                    if cur > max_punct_run: max_punct_run = cur
                    cur = 0
            max_punct_run = max(max_punct_run, cur)
            # sentence stats
            sents = [seg.strip() for seg in re.split(r"[.!?]+", s) if seg.strip()]
            sent_lens = [len(seg.split()) for seg in sents] if sents else []
            mean_sent = np.mean(sent_lens) if sent_lens else 0.0
            var_sent  = np.var(sent_lens) if sent_lens else 0.0
            short_sent_ratio = self._safe(sum(l<=7 for l in sent_lens), len(sent_lens)) if sent_lens else 0.0
            # shapes
            shapes = word_shapes(words)
            cap_all_ratio = self._safe(sum(w.isupper() for w in re.findall(r"\b\w+\b", s)), n_words)
            shape_xxxx_ratio = self._safe(sum(sh=="xxxx" for sh in shapes), len(shapes)) if shapes else 0.0
            # compressibility
            gz = gzip_ratio(s)
            feats.append([
                chars, n_words, avg_word_len, long_word_ratio, ttr, hapax,
                self._safe(punct_cnt, chars), self._safe(digit_cnt, chars),
                self._safe(upper_cnt, chars), self._safe(space_cnt, chars),
                self._safe(func_cnt, n_words), ent,
                rep1, rep2, rep3, uniq1, uniq2, uniq3,
                max_tok_run, max_punct_run, mean_sent, var_sent, short_sent_ratio,
                cap_all_ratio, shape_xxxx_ratio, gz
            ])
        return np.asarray(feats, dtype=np.float32)
    def get_feature_names_out(self, input_features=None):
        return np.array([
            "n_chars","n_words","avg_word_len","long_word_ratio","type_token_ratio","hapax_ratio",
            "punct_ratio","digit_ratio","upper_ratio","space_ratio","function_ratio","char_entropy",
            "rep1_ratio","rep2_ratio","rep3_ratio","uniq1_ratio","uniq2_ratio","uniq3_ratio",
            "max_token_run","max_punct_run","mean_sent_len","var_sent_len","short_sent_ratio",
            "cap_all_ratio","shape_xxxx_ratio","gzip_ratio"
        ], dtype=object)

In [8]:
# =========================
# 6) Vectorizers & Union (hybrid)
# =========================
word_tfidf = TfidfVectorizer(
    analyzer="word", ngram_range=(1,2),
    min_df=3, max_df=0.90, max_features=70000,
    sublinear_tf=True, strip_accents="unicode", lowercase=True,
    preprocessor=normalize_text, dtype=np.float32
)
char_tfidf = TfidfVectorizer(
    analyzer="char", ngram_range=(2,6),
    min_df=3, max_df=1.0, max_features=70000,
    sublinear_tf=True, lowercase=False, dtype=np.float32
)
char_hash = HashingVectorizer(
    analyzer="char", ngram_range=(3,7),
    n_features=2**18, alternate_sign=False, norm="l2",
    lowercase=False, dtype=np.float32
)
numeric_pipe = Pipeline([
    ("stylo", StylisticFeatures()),
    ("scale", StandardScaler(with_mean=False))
])
svd_dim = 128
word_svd_pipe = Pipeline([
    ("tfidf", clone(word_tfidf)),
    ("svd", TruncatedSVD(n_components=svd_dim, random_state=RANDOM_STATE))
])
features = FeatureUnion(
    transformer_list=[
        ("word", word_tfidf),
        ("char", char_tfidf),
        ("charh", char_hash),
        ("num",  numeric_pipe),
        ("wsvd", word_svd_pipe),
    ],
    transformer_weights={"word":0.6, "char":1.0, "charh":0.8, "num":1.5, "wsvd":0.7},
    n_jobs=1
)

<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [9]:
# =========================
# 7) Fit HYBRID on train, transform val
# =========================
X_train = features.fit_transform(train["text"].astype(str).values)
y_train = train["label"].values
X_val   = features.transform(val["text"].astype(str).values)
y_val   = val["label"].values

# Prior reweighting (train prior -> val prior)
pi_tr, pi_va = y_train.mean(), y_val.mean()
w1 = pi_va / max(1e-6, pi_tr); w0 = (1-pi_va) / max(1e-6, 1-pi_tr)
sw = np.where(y_train==1, w1, w0)

hybrid_base = LogisticRegression(
    solver="saga", penalty="elasticnet", l1_ratio=0.25,
    C=1.0, class_weight=None, max_iter=500, n_jobs=JOBS,
    random_state=RANDOM_STATE
).fit(X_train, y_train, sample_weight=sw)

hybrid_cal = CalibratedClassifierCV(hybrid_base, method="isotonic", cv="prefit").fit(X_val, y_val)
val_p_hybrid = hybrid_cal.predict_proba(X_val)[:,1]

ValueError: Only sparse matrices with 32-bit integer indices are accepted. Got int64 indices.

In [ ]:
# =========================
# 8) Specialist models: word-only, char-only, numeric-only
# =========================
# Build branch-specific pipelines freshly (avoid reusing fitted clones)
word_only_vec = clone(word_tfidf)
char_only_vec = clone(char_tfidf)
num_only_pipe = Pipeline([("stylo", StylisticFeatures()), ("scale", StandardScaler(with_mean=False))])

def fit_branch(train_text, y, val_text, vectorizer_or_pipe):
    # Make a simple LR for the branch
    Xtr = vectorizer_or_pipe.fit_transform(train_text)
    Xva = vectorizer_or_pipe.transform(val_text)
    clf = LogisticRegression(
        solver="saga", penalty="elasticnet", l1_ratio=0.15,
        C=1.0, class_weight=None, max_iter=400, n_jobs=JOBS,
        random_state=RANDOM_STATE
    ).fit(Xtr, y)
    cal = CalibratedClassifierCV(clf, method="isotonic", cv="prefit").fit(Xva, y_val)
    return vectorizer_or_pipe, clf, cal, cal.predict_proba(Xva)[:,1]

# Fit branches
word_vec_f,  word_clf_f,  word_cal_f,  val_p_word  = fit_branch(train["text"].astype(str), y_train, val["text"].astype(str), word_only_vec)
char_vec_f,  char_clf_f,  char_cal_f,  val_p_char  = fit_branch(train["text"].astype(str), y_train, val["text"].astype(str), char_only_vec)
num_pipe_f,  num_clf_f,   num_cal_f,   val_p_num   = fit_branch(train["text"].astype(str), y_train, val["text"].astype(str), num_only_pipe)

In [ ]:
# =========================
# 9) META-STACK: fit on validation probabilities
# =========================
val_lengths = val["text"].astype(str).str.len().values
X_meta_val = np.vstack([val_p_hybrid, val_p_word, val_p_char, val_p_num, (val_lengths/1000.0)]).T
meta = LogisticRegression(solver="lbfgs", C=2.0, max_iter=200, random_state=RANDOM_STATE).fit(X_meta_val, y_val)
val_p_meta = meta.predict_proba(X_meta_val)[:,1]

# Bucketed thresholds on META probs (length-aware)
length_split = int(np.quantile(val_lengths, 0.5))
grid = np.linspace(0.15, 0.85, 71)
best = {"thr_lo":0.5,"thr_hi":0.5,"f1":-1.0}
for thr_lo in grid:
    for thr_hi in grid:
        preds = np.where(val_lengths < length_split,
                         (val_p_meta >= thr_lo).astype(int),
                         (val_p_meta >= thr_hi).astype(int))
        f1 = f1_score(y_val, preds)
        if f1 > best["f1"]:
            best = {"thr_lo":thr_lo, "thr_hi":thr_hi, "f1":f1}

_print_section("Validation results (stacked)")
try:
    print("Hybrid AUROC:", roc_auc_score(y_val, val_p_hybrid))
    print("Word   AUROC:", roc_auc_score(y_val, val_p_word))
    print("Char   AUROC:", roc_auc_score(y_val, val_p_char))
    print("Num    AUROC:", roc_auc_score(y_val, val_p_num))
    print("META   AUROC:", roc_auc_score(y_val, val_p_meta))
except Exception: pass
print(f"Best F1 (META, bucketed): {best['f1']:.4f} | thr_lo={best['thr_lo']:.3f} | thr_hi={best['thr_hi']:.3f}")
val_pred_bucketed = np.where(val_lengths < length_split,
                             (val_p_meta >= best["thr_lo"]).astype(int),
                             (val_p_meta >= best["thr_hi"]).astype(int))
print("\nValidation report @META bucketed:")
print(classification_report(y_val, val_pred_bucketed, digits=4))

In [ ]:
# =========================
# 10) FINAL FIT on train+val and TEST INFERENCE (stacked)
# =========================
# Refit HYBRID on train+val with prior reweighting to val prior (operating point consistency)
X_trainval = features.fit_transform(pd.concat([train["text"], val["text"]]).astype(str).values)
y_trainval = np.concatenate([y_train, y_val])
pi_trv = y_trainval.mean()
w1 = pi_va / max(1e-6, pi_trv); w0 = (1-pi_va) / max(1e-6, 1-pi_trv)
sw_trv = np.where(y_trainval==1, w1, w0)
hybrid_final = LogisticRegression(
    solver="saga", penalty="elasticnet", l1_ratio=0.25,
    C=1.0, class_weight=None, max_iter=600, n_jobs=JOBS,
    random_state=RANDOM_STATE
).fit(X_trainval, y_trainval, sample_weight=sw_trv)

# Refit branches on train+val
word_vec_F = clone(word_tfidf); word_vec_F.set_params(preprocessor=normalize_text)
char_vec_F = clone(char_tfidf)
num_pipe_F = Pipeline([("stylo", StylisticFeatures()), ("scale", StandardScaler(with_mean=False))])

X_test = features.transform(test["text"].astype(str).values)
p_hybrid_test = hybrid_final.predict_proba(X_test)[:,1]

# Branch predictions
def fit_branch_tv(trainval_text, y_tv, test_text, vectorizer_or_pipe):
    X_tv = vectorizer_or_pipe.fit_transform(trainval_text)
    X_te = vectorizer_or_pipe.transform(test_text)
    clf = LogisticRegression(
        solver="saga", penalty="elasticnet", l1_ratio=0.15,
        C=1.0, class_weight=None, max_iter=500, n_jobs=JOBS,
        random_state=RANDOM_STATE
    ).fit(X_tv, y_tv)
    return clf.predict_proba(X_te)[:,1]

p_word_test = fit_branch_tv(pd.concat([train["text"], val["text"]]).astype(str), y_trainval, test["text"].astype(str), word_vec_F)
p_char_test = fit_branch_tv(pd.concat([train["text"], val["text"]]).astype(str), y_trainval, test["text"].astype(str), char_vec_F)
p_num_test  = fit_branch_tv(pd.concat([train["text"], val["text"]]).astype(str), y_trainval, test["text"].astype(str), num_pipe_F)

# META on test
test_lengths = test["text"].astype(str).str.len().values
X_meta_test  = np.vstack([p_hybrid_test, p_word_test, p_char_test, p_num_test, (test_lengths/1000.0)]).T
p_meta_test  = meta.predict_proba(X_meta_test)[:,1]
test_pred    = np.where(test_lengths < length_split,
                        (p_meta_test >= best["thr_lo"]).astype(int),
                        (p_meta_test >= best["thr_hi"]).astype(int))

submission = pd.DataFrame({"id": test["id"], "label": test_pred})
out_path = BASE_DIR + "submission_stacked_supercharged.csv"
submission.to_csv(out_path, index=False)

_print_section("Artifacts saved")
print("Saved predictions to:", out_path)

In [ ]:
# =========================
# 11) Quick interpretability: top weights from hybrid_final
# =========================
def feature_names_from_fitted_union(fu: FeatureUnion):
    names = []
    for name, trans in fu.transformer_list:
        if name in ("word","char"):
            if hasattr(trans, "get_feature_names_out"):
                names += [f"{name}::{t}" for t in trans.get_feature_names_out()]
        elif name=="num" and hasattr(trans, "named_steps"):
            sty = trans.named_steps.get("stylo")
            if sty is not None and hasattr(sty, "get_feature_names_out"):
                names += [f"num::{t}" for t in sty.get_feature_names_out()]
        elif name=="wsvd":
            names += [f"wsvd::svd_{i}" for i in range(128)]
        elif name=="charh":
            names += [f"charh::<hash>"]
    return np.array(names, dtype=object)

try:
    fn = feature_names_from_fitted_union(features)
    co = hybrid_final.coef_.ravel()
    k = min(30, co.size)
    top_pos = np.argsort(-co)[:k]; top_neg = np.argsort(co)[:k]
    _print_section("Hybrid top weights (+)")
    for i in top_pos: print(f"{fn[i] if i<len(fn) else i}\t{co[i]:.3f}")
    _print_section("Hybrid top weights (-)")
    for i in top_neg: print(f"{fn[i] if i<len(fn) else i}\t{co[i]:.3f}")
except Exception as e:
    print("Coefficient inspection skipped:", e)
